# Study 939 — DRIP or Sweep 💧

**Reinvest each dividend the day it lands, or sweep it to cash and reinvest on a schedule?**

Every broker's help page tells you to tick the automatic-reinvestment box, because idle
cash is cash not compounding. That is a testable claim about a *timing difference on a
small cash flow*, and nobody who repeats it seems to have put a number on it.

We put one on it. Two investors, the **same fund**, the **same shares on day one**.
**DRIP** buys more shares on the pay date. **SWEEP** parks the cash in T-bills (BIL) and
buys at the next quarter end. Everything else — one execution lag, one-way costs on the
amount traded, excess-of-cash Sharpes — is identical.

Tapes: **SPY, VYM, SCHD** against **BIL**, 2007-05-30 → 2026-06-30
(4,801 days; SCHD from 2011). The distribution stream is **reconstructed** from
the price-only and total-return legs.

*Real numbers below are the frozen headline (`docs/results.md`, Fingerprint
`138ed0f07aec`, as-of 2026-06-30); the live cells run the offline synthetic control only.*


## 1. Where the money can even come from

A dividend leaves the share price on the **ex-date** and lands in your account on the **pay date**, two to five weeks later. From that moment it is either buying shares (DRIP) or sitting in T-bills waiting for the calendar (SWEEP).

So the entire prize is: *what the fund earns over cash, on that money, for the few weeks the sweeper is waiting*. Multiply it out — a 2% yield, a 5-point equity-over-cash spread, a six-week wait — and you get about **one basis point a year**. That is the size of the thing before we touch any data.

> 🔬 **For the quants:** the gap is (distribution yield) × (realised equity-minus-cash return over the delay) × (average delay). Everything else in this study is measurement error around that product.

## 2. What the tape actually pays

Nineteen years of SPY, VYM and SCHD, with the dividend stream rebuilt from the price and total-return series and every reinvestment costed:

In [1]:
R = {'spy': {'years': 19.1, 'sh_drip': 0.542, 'sh_sweep': 0.5424, 'w_drip': 69003, 'w_sweep': 68729, 'gap': 2.09, 't': 1.09, 'ci': (-1.18, 4.99), 'yld': 1.86}, 'vym': {'years': 19.1, 'sh_drip': 0.4865, 'sh_sweep': 0.4867, 'w_drip': 51370, 'w_sweep': 51033, 'gap': 3.46, 't': 1.18, 'ci': (-1.75, 7.97), 'yld': 3.12}, 'schd': {'years': 14.7, 'sh_drip': 0.7785, 'sh_sweep': 0.7801, 'w_drip': 60195, 'w_sweep': 59863, 'gap': 3.78, 't': 1.41, 'ci': (-1.74, 8.3), 'yld': 3.2}, 'matched': {'SPY': (1.77, 3.01, 1.87, 1.7), 'VYM': (3.09, 3.73, 1.4, 1.21), 'SCHD': (3.2, 3.78, 1.41, 1.18)}}
for tag, d in [('SPY ', R['spy']), ('VYM ', R['vym']), ('SCHD', R['schd'])]:
    print(f"{tag} (realised yield {d['yld']:.2f}%): DRIP ends at {d['w_drip']:,} vs "
          f"SWEEP {d['w_sweep']:,} on 10,000 invested over {d['years']} years")
    print(f"       -> DRIP wins by {d['gap']:+.2f} basis points a year "
          f"(HAC t = {d['t']:+.2f})")

SPY  (realised yield 1.86%): DRIP ends at 69,003 vs SWEEP 68,729 on 10,000 invested over 19.1 years
       -> DRIP wins by +2.09 basis points a year (HAC t = +1.09)
VYM  (realised yield 3.12%): DRIP ends at 51,370 vs SWEEP 51,033 on 10,000 invested over 19.1 years
       -> DRIP wins by +3.46 basis points a year (HAC t = +1.18)
SCHD (realised yield 3.20%): DRIP ends at 60,195 vs SWEEP 59,863 on 10,000 invested over 14.7 years
       -> DRIP wins by +3.78 basis points a year (HAC t = +1.41)


## 3. Three basis points, in money

On **10,000** invested, ticking the DRIP box instead of sweeping quarterly was worth **274 over 19.1 years on SPY** — about **14 a year**. On the dividend funds it is a bit more: **18/yr** (VYM) and **23/yr** (SCHD). That is less than the half-spread on a single ETF trade.

It is tempting to read the ordering — 2.09 bps on the 1.86%-yield SPY, 3.46 on VYM, 3.78 on SCHD — as the mechanism showing its face: more cash in transit, more to lose by parking it. Careful. SPY and VYM are measured over 2007-2026 and SCHD only over 2011-2026, and when all three are re-raced on the *same* window the spread nearly disappears (3.01 / 3.73 / 3.78 bps/yr). The mechanism is real arithmetic; the cross-fund pattern is mostly calendar.

## 4. The one version that costs real money

Sweeping **quarterly** costs almost nothing, because the pay-date lag has already eaten most of the delay by the time the quarter ends. Sweeping **annually** — letting a year of distributions pile up in cash before reinvesting — is a different story:

| Fund | quarterly sweep | annual sweep |
|---|--:|--:|
| SPY | +2.09 bps/yr | **+9.18** bps/yr |
| VYM | +3.46 bps/yr | **+15.46** bps/yr |
| SCHD | +3.78 bps/yr | **+17.77** bps/yr |

Still small — but 15 to 18 basis points a year is at least the size of a fund fee, and it is the only cut in this study where the numbers clear the desk's significance bar (SCHD *t* = +2.73).

## 5. The number we cannot see

Here is the honest problem. Yahoo publishes the **ex-date**; nobody free publishes the **pay date**. We assumed 30 calendar days. Change that assumption and the answer moves like this (SCHD):

| assumed pay lag | 0 days | 15 days | 30 days | 45 days |
|---|--:|--:|--:|--:|
| gap | +0.80 | +6.96 | +3.78 | +2.47 bps/yr |

**The thing we had to guess moves the answer more than the answer is worth.** That is the sentence to remember from this study.

## 6. Live check — is the measuring stick straight? (offline synthetic)

Before believing a two-basis-point result, check the machinery can find a *planted* one and stays quiet when there is nothing there. On a deliberately loud laboratory tape (a big premium over cash, a fat dividend, low noise) the DRIP arm must win; on a tape where the fund drifts at exactly the cash rate, parking costs nothing and the gap must vanish.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from drip_sweep import data, strategy as st
planted = st.seed_sweep(data.synthetic_daily, 1.0, n_seeds=8)
null    = st.seed_sweep(data.synthetic_daily, 0.0, n_seeds=8)
print('lab tape, planted premium : gap %+.2f bps/yr (se %.2f)  <- must be clearly positive'
      % (planted['mean'], planted['se']))
print('lab tape, no premium      : gap %+.2f bps/yr (se %.2f)  <- must be ~0'
      % (null['mean'], null['se']))

lab tape, planted premium : gap +14.23 bps/yr (se 0.56)  <- must be clearly positive
lab tape, no premium      : gap +0.03 bps/yr (se 0.60)  <- must be ~0


## 7. And the check that decides the verdict

Now feed the *same* measuring stick a **market-realistic** world — a 5.5% premium, a 3% yield, 16% volatility — and ask it to find the true effect in twenty years of daily data. It cannot. The noise from one twenty-year path is bigger than the thing being measured.

In [3]:
real = dict(equity_premium=0.055, div_yield_ann=0.03, vol_ann=0.16)
pl = st.seed_sweep(data.synthetic_daily, 1.0, n_seeds=8, gen_kw=real)
nl = st.seed_sweep(data.synthetic_daily, 0.0, n_seeds=8, gen_kw=real)
print('realistic world, true effect present: %+.2f bps/yr (se %.2f)' % (pl['mean'], pl['se']))
print('realistic world, no effect         : %+.2f bps/yr (se %.2f)' % (nl['mean'], nl['se']))
print('\nOne 20-year tape cannot tell these two apart. That is why the real-tape')
print('t-statistic is ~1.2 and not ~5: the effect is real and it is tiny.')

realistic world, true effect present: +1.50 bps/yr (se 0.80)
realistic world, no effect         : -0.52 bps/yr (se 0.81)

One 20-year tape cannot tell these two apart. That is why the real-tape
t-statistic is ~1.2 and not ~5: the effect is real and it is tiny.


## Verdict

- **Signal — Weak.** DRIP wins on all three funds and in both eras (+2.09 / +3.46 / +3.78 bps/yr) — the right sign everywhere. But the *t*-statistics are +1.09 / +1.18 / +1.41, every confidence interval includes zero, the yield ordering mostly evaporates once the three funds are raced on the same window, and the unobservable pay-date lag swings the estimate across a wider band than the estimate itself. Only the annual-sweep variant clears the desk's |*t*| ≥ 2 bar.
- **Tradability — Mirage.** +2.81 basis points a year is 14-23 a year on 10,000 — less than one trade's spread. Tick the DRIP box because it is free and saves you from making a decision four times a year. If you prefer to sweep, do it **quarterly, not annually**: that is the only version of this choice that costs measurable money.